<a href="https://colab.research.google.com/github/Matheusbcy/-Data-Science-IA-/blob/main/Atendimento_e_Suporteipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests==2.32.4
!pip install langchain langchain-groq langchain_community langchain-huggingface --q
!pip install faiss-cpu sentence_transformers PyMuPDF --q

In [ ]:
!pip install -q streamlit python-dotenv
!pip install -q localtunnel

In [ ]:
!pip install pyngrok

In [14]:
from pyngrok import ngrok

In [7]:
%%writefile .env
GROQ_API_KEY = SUA CHAVE GROQ

Writing .env


Crie sua conta aqui e crie sua key (lembre-se de salvar a key assim que criar )  
https://groq.com/

In [55]:
%%writefile app02.py
import streamlit as st
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from dotenv import load_dotenv

load_dotenv()

st.set_page_config(page_title = "Atendimento SafeBank")
st.title("Atendimento SafeBank")

id_model = "deepseek-r1-distill-llama-70b"
temperature = 0.7
path = "/content"


### Carregamento da LLM
def load_llm(id_model, temparature):
  llm = ChatGroq(
      model = id_model,
      temperature = temparature,
      max_tokens = None,
      timeout = None,
      max_retries = 2
  )
  return llm

llm = load_llm(id_model, temperature)

### Exibição do resultado
def show_res(res):
  from IPython.display import Markdown
  if "</think>" in res:
    res = res.split("</think>")[-1].strip()
  else:
    res = res.strip()
  display(Markdown(res))

### Extração do conteúdo
def extract_text_pdf(file_path):
  loader = PyMuPDFLoader(file_path)
  doc = loader.load()
  content = "\n".join([page.page_content for page in doc])
  return content

### Indexação e recuperação

def config_retriever(folder_path = "/content"):
  # Carregar documentos
  docs_path = Path("/content")
  pdf_files = [f for f in docs_path.glob("*.pdf")]

  loaded_documents = [extract_text_pdf(pdf) for pdf in pdf_files]

  # Divisão em pedaços de texto
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = 1000,
      chunk_overlap = 200
  )
  chunks = []
  for doc in loaded_documents:
    chunks.extend(text_splitter.split_text(doc))

  #Embeddings
  embeddings_model = "BAAI/bge-m3"

  embeddings = HuggingFaceEmbeddings(model_name = embeddings_model)

  # Armazenamento
  vectorstore = FAISS.from_texts(chunks, embedding = embeddings)

  # Configurando o recuperador de texto

  retriever = vectorstore.as_retriever(
      search_type = "mmr",
      search_kwargs = {"k": 3, "fetch_k": 4}
  )

  return retriever


def config_rag_chain(llm, retriever):
  context_q_system_prompt = "Given the following chat history and the follow-up question which might reference context in the chat history, formulate a standalone question wich can be understood without the chat history. Do NOT answer the question, just reformulate it if needed and otherwise return it as is."
  context_q_system_prompt = context_q_system_prompt
  context_q_user_prompt = "Question: {input}"
  contexto_q_prompt = ChatPromptTemplate.from_messages(
      [
          ("system", context_q_system_prompt),
          ("human", context_q_user_prompt)
      ]
  )
  # Chain para contextualização
  history_aware_retriever = create_history_aware_retriever(
    llm = llm,
    retriever = retriever,
    prompt = contexto_q_prompt
  )

  # Prompt para perguntas e respostas (Q&A)
  prompt_template = """Use o seguinte contexto e histórico de conversa para responder a pergunta:

  Contexto: {context}

  Histórico da conversa:
  {chat_history}

  Pergunta: {input}

  Resposta:"""

  qa_prompt = ChatPromptTemplate.from_messages([
    ("system", prompt_template),
    MessagesPlaceholder("chat_history"),
    ("human", "Pergunta: {input}\nContexto: {context}"),
  ])

  qa_chain = create_stuff_documents_chain(llm, qa_prompt)

  rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

  return rag_chain

def chat_llm(rag_chain, input):
  st.session_state.chat_history.append(HumanMessage(content = input))

  response = rag_chain.invoke({
      "input": input,
      "chat_history": st.session_state.chat_history
  })

  res = response["answer"]
  res = res.split("</think>")[-1].strip() if "</think>" in res else res.strip()

  st.session_state.chat_history.append(AIMessage(content = res))

  return res

# Inicializações da session_state
if "chat_history" not in st.session_state:
    st.session_state.chat_history = [
        AIMessage(content = "Olá, sou seu assistente virtual! Como posso te ajudar?")
    ]

if "retriever" not in st.session_state:
    st.session_state.retriever = None

if "chat_started" not in st.session_state:
    st.session_state.chat_started = False

if "loading" not in st.session_state:
    st.session_state.loading = False

# Tela de carregamento
if st.session_state.loading:
    with st.container():
        col1, col2, col3 = st.columns([1, 2, 1])
        with col2:
            st.info("🔄 Aguarde enquanto te transferimos para um atendente...")
            with st.spinner("Carregando documentos e preparando o sistema..."):
                # Configura o retriever (processamento dos documentos)
                if st.session_state.retriever is None:
                    st.session_state.retriever = config_retriever(path)
                # Finaliza o loading
                st.session_state.loading = False
                st.rerun()

# Chat normal (após carregamento completo)
elif st.session_state.chat_started and not st.session_state.loading:
    # Exibe o histórico de chat apenas se o atendimento foi iniciado
    for message in st.session_state.chat_history:
        if isinstance(message, AIMessage):
            with st.chat_message("AI"):
                st.write(message.content)
        elif isinstance(message, HumanMessage):
            with st.chat_message("Human"):
                st.write(message.content)

# Input do usuário
user_input = st.chat_input("Digite sua pergunta aqui...")

if user_input:
    with st.chat_message("Human"):
        st.markdown(user_input)

    with st.chat_message("AI"):
        rag_chain = config_rag_chain(llm, st.session_state.retriever)
        res = chat_llm(rag_chain, user_input)
        st.write(res)

# Tela inicial (antes de clicar no botão)
elif not st.session_state.chat_started and not st.session_state.loading:
    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        if st.button("🚀 Iniciar atendimento", type="primary", use_container_width=True):
            st.session_state.loading = True
            st.session_state.chat_started = True
            st.rerun()

Overwriting app02.py


## Execução streamlit

In [ ]:
!streamlit run app02.py &>/content/logs.txt &
!wget -q -O - ipv4.icanhazip.com
!npx localtunnel --port 8501

Crie sua conta aqui e crie sua key (lembre-se de salvar a key assim que criar )  
https://ngrok.com/

In [59]:
!ngrok config add-authtoken SUA CHAVE NGROK AQUI
!streamlit run app02.py --server.port 8501 &>/content/logs.txt &

public_url = ngrok.connect(8501)
public_url

ERROR:  accepts 1 arg(s), received 4


<NgrokTunnel: "https://86f110603528.ngrok-free.app" -> "http://localhost:8501">

Recomendo usar o Ngrok por ser mais robusto e compatível com Streamlit.